# Average, Sample Standard Deviation, Sample Correlation

In this section we will be exploring estimators for the expectation, standard deviation and correlation and how accurate they are.

Let's start by looking at the normal distribution. In the code below we calculate the sample average and sample standard deviation and compare it to the true parameters. We sample from the normal distribution using numpy by calling `normal()`.

In [ ]:
import micropip

await micropip.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, fixed

In [ ]:
def update_plots(expectation_mu, standard_dev, n, num_samples):
    estimates_expectation = []
    estimates_standard_deviation = []
    errors_expectation = []
    errors_standard_deviation = []

    # simulate samples
    for i in range(num_samples):
        rng = np.random.default_rng()
        sample = rng.normal(loc=expectation_mu, scale=standard_dev, size=n)

        expectation = sample.sum() / n
        standard_deviation = np.std(sample, ddof=1)
        errors_expectation.append(expectation_mu - expectation)
        errors_standard_deviation.append(standard_dev - standard_deviation)
        estimates_expectation.append(expectation)
        estimates_standard_deviation.append(standard_deviation)

    estimates_exp = np.array(estimates_expectation)
    mean_exp = estimates_exp.mean()
    estimates_std = np.array(estimates_standard_deviation)
    mean_std = estimates_std.mean()

    errors_exp = np.array(errors_expectation)
    mse_exp = np.mean(errors_exp**2)

    errors_std = np.array(errors_standard_deviation)
    mse_std = np.mean(errors_std**2)

    print(f"MSE for mu = {mse_exp:.4f}")
    print(f"MSE for sigma = {mse_std:.4f}")

    # plot estimates
    plt.figure(figsize=(15, 9))
    plt.subplot(1, 2, 1)
    plt.hist(estimates_exp)
    plt.title("Estimation $\hat{\mu}$")
    plt.ylabel("frequency")
    plt.xlabel("$\mu$ estimates")
    plt.xlim(2.2, 2.8)

    plt.axvline(x=expectation_mu, color="r", linestyle="--")
    plt.axvline(x=mean_exp, color="y", linestyle="--")
    plt.legend(
        [f"true value = {expectation_mu}", f"mean of estimates = {mean_exp:.2f}"]
    )

    plt.subplot(1, 2, 2)
    plt.hist(estimates_std)
    plt.title("Estimation $\sigma$")
    plt.ylabel("frequency")
    plt.xlabel("$\sigma$ estimates")
    plt.xlim(0.8, 1.2)

    plt.axvline(x=standard_dev, color="r", linestyle="--")
    plt.axvline(x=mean_std, color="y", linestyle="--")
    plt.legend([f"true value = {standard_dev}", f"mean of estimates = {mean_std:.2f}"])

    plt.show()


mu_true = 2.5
samples = 100
sd = 1.0

slider_n = widgets.IntSlider(
    value=100, min=100, max=1000, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plots,
    expectation_mu=fixed(mu_true),
    standard_dev=fixed(sd),
    n=slider_n,
    num_samples=fixed(samples),
)
interactive_plot

Observe how the MSE and estimates are impacted by the sample size.

Next, we will be looking at estimating the sample correlation given a multivariate normal distribution. Run the code below to calculate the sample correlation and compare it to the actual value.

We sample from a multivariate normal distribution by passing the means and covariance of our distribution to numpy's `random.Generator.multivariate_normal` function. We calculate the Pearson's correlation coefficient using `np.corrcoef`.

In [ ]:
def update_plots_multivariate(means, covariance, n, num_samples):
    rng = np.random.default_rng()
    estimates = []
    errors = []
    true_correlation = covariance[0][1] / (
        np.sqrt(covariance[0][0]) * np.sqrt(covariance[1][1])
    )

    # simulate samples
    for i in range(num_samples):
        sample = rng.multivariate_normal(means, covariance, size=n)
        sample_correlation = np.corrcoef(sample, rowvar=False)[0, 1]
        errors.append(true_correlation - sample_correlation)
        estimates.append(sample_correlation)

    estimates = np.array(estimates)
    mean_est = estimates.mean()

    errors = np.array(errors)
    mse = np.mean(errors**2)

    print(f"MSE for correlation = {mse:.4f}")

    # plot estimates
    plt.figure(figsize=(15, 9))
    plt.hist(estimates)
    plt.title("Estimation Correlation Coefficient")
    plt.ylabel("frequency")
    plt.xlabel("$estimates")
    plt.axvline(x=true_correlation, color="r", linestyle="--")
    plt.axvline(x=mean_est, color="y", linestyle="--")
    plt.legend(
        [f"true value = {true_correlation:.2f}", f"mean of estimates = {mean_est:.2f}"]
    )

    plt.show()


cov = 1.0
var_x = 0.5
var_y = 4.5
mu_true = 0.5

samples = 100
slider_n = widgets.IntSlider(
    value=100, min=100, max=1000, step=10, description="Sample Size"
)

interactive_plot = interactive(
    update_plots_multivariate,
    means=fixed([mu_true, mu_true]),
    covariance=fixed([[var_x, cov], [cov, var_y]]),
    n=slider_n,
    num_samples=fixed(samples),
)
interactive_plot